# Эксперимент 35: ASR for pronunciation diagnosis

**Статья:** Automatic speech recognition for diagnosis of pronunciation issues in children with speech sound disorders (Автоматическое распознавание речи для диагностики произносительных нарушений у детей с речевыми звуковыми нарушениями) 2024

**Ссылка:** [https://pubmed.ncbi.nlm.nih.gov/39162064/](https://pubmed.ncbi.nlm.nih.gov/39162064/)

**Краткое описание модели:** Pronunciation diagnostics из CTC-потока: confidence/entropy + temporal consistency токенов + классификатор.

**Содержание статьи:** Пайплайн ориентирован на диагностику произношения через устойчивость и уверенность ASR-декодирования, что ближе к задаче pronunciation diagnosis.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from difflib import SequenceMatcher
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

I0000 00:00:1774800354.944521   48191 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Загрузка данных и разбиение

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
n_letters = letters_train.shape[1]

Train: 1942, Val: 417, Test: 417


## 2. Подготовка признаков / представлений

In [ ]:
MODEL_ID = "facebook/wav2vec2-base-960h"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_ID).to(device)
model.eval()
blank_id = model.config.pad_token_id

def collapse(ids):
    out=[]; prev=None
    for i in ids:
        if i!=prev and (blank_id is None or i!=blank_id):
            out.append(int(i))
        prev=i
    return out

def extract_diag(path):
    y, sr = data_utils.load_audio(path, sr=16000, max_sec=config.MAX_DURATION_SEC)
    inp = processor(y, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(inp.input_values.to(device)).logits[0]
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1)
    token_div = len(set(collapse(pred))) / max(1, len(collapse(pred)))

    win_s, hop_s = int(2.0*16000), int(1.0*16000)
    seqs=[]
    if len(y) <= win_s:
        seqs=[collapse(pred)]
    else:
        for st in range(0, max(1, len(y)-win_s+1), hop_s):
            chunk=y[st:st+win_s]
            cinp=processor(chunk, sampling_rate=16000, return_tensors="pt", padding=True)
            with torch.no_grad():
                clogits=model(cinp.input_values.to(device)).logits[0]
            cids=torch.argmax(clogits, dim=-1).cpu().numpy()
            seqs.append(collapse(cids))
    sims=[]
    for a,b in zip(seqs[:-1], seqs[1:]):
        sims.append(SequenceMatcher(None, ' '.join(map(str,a)), ' '.join(map(str,b))).ratio())
    if not sims:
        sims=[1.0]

    return np.array([
        conf.mean(), conf.std(), conf.min(),
        entropy.mean(), entropy.std(), entropy.max(),
        float(np.mean(pred == blank_id)) if blank_id is not None else 0.0,
        float(token_div),
        float(np.mean(sims)), float(np.std(sims)), float(np.min(sims)),
    ], dtype=np.float32)

X_train = np.stack([extract_diag(p) for p in paths_train])
X_val   = np.stack([extract_diag(p) for p in paths_val])
X_test  = np.stack([extract_diag(p) for p in paths_test])

X_train = np.hstack([X_train, letters_train])
X_val   = np.hstack([X_val, letters_val])
X_test  = np.hstack([X_test, letters_test])

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 3. Обучение, оценка и сохранение результата

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=config.RANDOM_STATE))])
param_grid = {"clf__C": [0.3, 1.0, 3.0], "clf__gamma": ["scale", "auto"]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
clf = grid.best_estimator_

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
display(pd.DataFrame([{"accuracy": accuracy, "f1_macro": f1_macro, "f1_bad": f1_bad, "roc_auc": roc_auc, "precision_bad": precision_bad, "recall_bad": recall_bad}]))

save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_35_asr_pronunciation_diagnosis", 
    experiment_name="ASR pronunciation diagnostics", 
    model="CTC pronunciation diagnostics + SVM", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes=f"ctc_pronunciation_diagnostics/grid={grid.best_params_}", 
    num_params=None, 
    train_time_sec=None
)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
              precision    recall  f1-score   support

        good       0.81      0.72      0.76       282
         bad       0.53      0.64      0.58       135

    accuracy                           0.70       417
   macro avg       0.67      0.68      0.67       417
weighted avg       0.72      0.70      0.70       417



,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad
0,0.697842,0.672022,0.58,0.775335,0.527273,0.644444


Результат сохранён в result.csv текущего эксперимента
